# 07 · Lineage check

The job's last task (#33). Confirms, from Unity Catalog lineage, that:
- every gold table reads only from `hawkerreach.silver` and `hawkerreach.gold`
- the app's service principal reads only `hawkerreach.gold`

It logs what it finds to `data_quality_log` and never fails the run. If system tables aren't available, it falls back to a static check that every app query file references only `hawkerreach.gold.*`; the static check runs either way.

In [0]:
import glob, json, os, re
from pyspark.sql import functions as F
from dq import DQLog  # dq.py at the repo root

CATALOG= "hawkerreach"
dbutils.widgets.text("app_name", "hawkerreach")
dbutils.widgets.text("lookback_days", "30")
APP= dbutils.widgets.get("app_name")
DAYS= int(dbutils.widgets.get("lookback_days"))
ALLOWED_GOLD_SOURCES= {f"{CATALOG}.silver", f"{CATALOG}.gold"}
APP_SCHEMA= f"{CATALOG}.gold"
dq= DQLog(spark, "07_lineage_check")
summary= {}

In [0]:
# 1. gold tables read only from silver and gold
try:
    lineage= spark.table("system.access.table_lineage").filter(F.col("event_date") >= F.date_sub(F.current_date(), DAYS))
    gold_reads= (lineage.filter((F.col("target_table_catalog") == CATALOG) & (F.col("target_table_schema") == "gold")
                                & F.col("source_table_full_name").isNotNull())
                 .select("target_table_full_name", "source_table_full_name").distinct())
    source_schema= F.regexp_extract("source_table_full_name", r"^([^.]+\.[^.]+)\.", 1)  # catalog.schema
    bad= gold_reads.filter(~source_schema.isin(*ALLOWED_GOLD_SOURCES)).collect()
    n_pairs= gold_reads.count()
    dq.issues("lineage_gold_sources", [(r["target_table_full_name"], f"reads {r['source_table_full_name']}") for r in bad])
    dq.step("lineage_gold_sources", n_pairs, n_pairs - len(bad), f"gold table <- source pairs in the last {DAYS} days that read only silver/gold")
    summary["gold_sources"]= {"pairs": n_pairs, "bad": [f"{r['target_table_full_name']} <- {r['source_table_full_name']}" for r in bad]}
    system_tables= True
except Exception as e:  # e.g. no system tables on this workspace
    print("system.access.table_lineage not available:", str(e)[:200])
    dq.issues("lineage_unavailable", [("system.access.table_lineage", str(e)[:500])])
    system_tables= False
summary["system_tables"]= system_tables

In [0]:
# 2. the app's service principal reads only gold
if system_tables:
    try:
        from databricks.sdk import WorkspaceClient
        sp= WorkspaceClient().apps.get(APP).service_principal_client_id
    except Exception as e:
        sp= None
        print(f"couldn't look up app {APP}:", str(e)[:200])
        dq.issues("lineage_app_lookup", [(APP, str(e)[:500])])
    if sp:
        app_reads= (lineage.filter((F.col("created_by") == sp) & F.col("source_table_full_name").isNotNull())
                    .select("source_table_full_name").distinct())
        reads= [r["source_table_full_name"] for r in app_reads.collect()]
        bad= [t for t in reads if not t.startswith(APP_SCHEMA + ".")]
        dq.issues("lineage_app_reads", [(t, f"app {APP} ({sp}) read a non-gold table") for t in bad])
        dq.step("lineage_app_reads", len(reads), len(reads) - len(bad), f"tables app {APP} read in the last {DAYS} days that are gold")
        summary["app_reads"]= {"service_principal": sp, "tables": sorted(reads), "bad": bad}

In [0]:
# 3. static fallback, run either way: every app query file references only hawkerreach.gold.*
files= sorted(glob.glob(os.path.join(os.getcwd(), "app", "config", "queries", "*.sql")))
refs= {}
for f in files:
    sql= re.sub(r"--[^\n]*", "", open(f).read())  # comments may name other tables
    refs[os.path.basename(f)]= sorted(set(re.findall(r"\b[a-z_]+\.[a-z_]+\.[a-z_]+\b", sql, flags=re.I)))
bad= [(name, t) for name, ts in refs.items() for t in ts if not t.lower().startswith(APP_SCHEMA + ".")]
if not files:
    print("no app query files found next to this notebook; the static check didn't run")
    dq.issues("lineage_static_queries", [("app/config/queries", "no query files synced with the job")])
else:
    dq.issues("lineage_static_queries", [(name, f"references {t}") for name, t in bad])
    dq.step("lineage_static_queries", len(files), len(files) - len({n for n, _ in bad}), "app query files that reference only hawkerreach.gold.*")
summary["static"]= {"files": len(files), "bad": [f"{n}: {t}" for n, t in bad]}

In [0]:
problems= len(summary.get("gold_sources", {}).get("bad", [])) + len(summary.get("app_reads", {}).get("bad", [])) + len(summary["static"]["bad"])
print("lineage check:", "OK" if problems == 0 else f"{problems} problem(s), logged to data_quality_log")
print(json.dumps(summary, indent=1))
dbutils.notebook.exit(json.dumps(summary))